In [0]:
%pip install databricks-ai-search
dbutils.library.restartPython()

In [0]:
import mlflow

SYSTEM_PROMPT = """You are PrivacyLens, an educational assistant for India's DPDP Act 2023 and the EU GDPR.
Rules:
1. Answer ONLY using the context provided. Do not use outside knowledge.
2. Cite every claim with its source in square brackets, exactly as labelled in the context, e.g. [GDPR Article 17] or [DPDP Section 12].
3. If the context does not contain the answer, reply exactly: "I could not find this in the GDPR or DPDP Act text provided."
4. When both laws are relevant, explain each and note key differences.
5. Keep the answer under 200 words. This is not legal advice."""

PROMPT_NAME = "workspace.privacylens.system_prompt"

prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=SYSTEM_PROMPT,
    commit_message="v1: grounded answers, bracket citations, fixed refusal sentence",
)
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=prompt.version)

print("Registered prompt version:", prompt.version)

In [0]:
loaded = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
print("Version:", loaded.version)
print(loaded.template)

In [0]:
import os, sys, importlib, mlflow, pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
folder = "/Workspace" + os.path.dirname(nb_path)
sys.path.append(folder)
importlib.invalidate_caches()

PROMPT_NAME = "workspace.privacylens.system_prompt"
MODEL_NAME = "workspace.privacylens.rag_model"

# 1. Read the production prompt from the registry
prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
print("Using prompt version:", prompt.version)

signature = ModelSignature(
    inputs=Schema([ColSpec("string", "question"), ColSpec("string", "law", required=False)]),
    outputs=Schema([ColSpec("string", "answer"), ColSpec("string", "sources")]),
)

# 2. Log the model with that prompt pinned inside it
with mlflow.start_run(run_name=f"privacylens_prompt_v{prompt.version}"):
    mlflow.log_param("prompt_version", prompt.version)
    info = mlflow.pyfunc.log_model(
        name="privacylens",
        python_model=f"{folder}/privacylens_model.py",
        code_paths=[f"{folder}/retrieval.py"],
        model_config={
            "system_prompt": prompt.template,
            "prompt_uri": f"prompts:/{PROMPT_NAME}/{prompt.version}",
        },
        signature=signature,
        input_example=pd.DataFrame({"question": ["Can a user make a company delete their data?"]}),
        pip_requirements=["mlflow", "databricks-ai-search", "pandas"],
        resources=[
            DatabricksVectorSearchIndex(index_name="workspace.privacylens.silver_chunks_index"),
            DatabricksServingEndpoint(endpoint_name="databricks-meta-llama-3-3-70b-instruct"),
            DatabricksServingEndpoint(endpoint_name="databricks-gte-large-en"),
        ],
    )

# 3. Register it in Unity Catalog and mark it as champion
mv = mlflow.register_model(info.model_uri, MODEL_NAME)
mlflow.MlflowClient().set_registered_model_alias(MODEL_NAME, "champion", mv.version)
print(f"Registered {MODEL_NAME} version {mv.version} as @champion")

In [0]:
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
out = champion.predict(pd.DataFrame({"question": ["Can a user make a company delete their data?"]}))
print("Sources:", out.loc[0, "sources"])
print("Answer:", out.loc[0, "answer"])